# 🏥 Hospital Virtual Assistant - Pipeline de Orquestração Clínica
> **Tech Challenge - Fase 3** | **Pós-Tech IA para Devs**
>
> **Grupo:** Grupo 5 (Kevin Makoto Shiroma) <br>
> **Data de Execução:** Maio de 2026

---

## 📌 Introdução e Objetivos do Projeto
Este notebook apresenta o desenvolvimento de um **Assistente Virtual Médico** integrado e seguro, capaz de auxiliar na tomada de decisões clínicas com base em dados de prontuários e protocolos internos.

A arquitetura foi projetada de forma **modular em Python** e utiliza o ecossistema **LangChain** para integração do modelo e **LangGraph** para a automação dos fluxos de governança e segurança.

### 🎯 Requisitos Clínicos e de Negócio Cobertos:

* **Governança via Nodes:** Desvio automático de fluxo e emissão de alertas caso existam exames pendentes.
* **Guardrails de Segurança:** Restrição absoluta contra automedicação ou prescrições diretas sem validação humana.
* **Auditabilidade:** Registro detalhado de logs de todas as transações para conformidade regulatória.

## 🛠️ Etapa 1: Estruturação e Modularização do Projeto
Para garantir as melhores práticas de engenharia de software e atender ao critério de modularização, criamos a árvore de diretórios nativa do backend da aplicação.

In [ ]:
!mkdir -p hospital_project/database hospital_project/models hospital_project/utils

## 🗄️ Etapa 2: Camada de Dados e Mecanismo de Recuperação (SQL Database & Tools)

Nesta etapa, inicializamos um banco de dados relacional SQLite que simula o sistema interno do hospital, contendo tabelas de pacientes, históricos clínicos e exames.

Também definimos a função fetch_patient_clinical_context decorada como uma @tool do LangChain. Ela atua como o nosso motor de RAG, extraindo o contexto atualizado de um paciente a partir do seu ID de forma estruturada.

In [ ]:
%%writefile hospital_project/database/connection.py
import sqlite3
import os
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool

def init_db():
    db_path = "/content/data.db"
    if os.path.exists(db_path):
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS patients (
            patient_id INTEGER PRIMARY KEY,
            synthetic_name TEXT NOT NULL,
            age INTEGER NOT NULL,
            gender TEXT NOT NULL,
            admission_date DATE NOT NULL
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS medical_records (
            record_id INTEGER PRIMARY KEY,
            patient_id INTEGER,
            record_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            chief_complaint TEXT,
            symptoms TEXT,
            diagnostic_hypothesis TEXT,
            FOREIGN KEY (patient_id) REFERENCES patients (patient_id)
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS exams (
            exam_id INTEGER PRIMARY KEY,
            patient_id INTEGER,
            exam_name TEXT NOT NULL,
            status TEXT NOT NULL,
            result TEXT,
            request_date DATE NOT NULL,
            FOREIGN KEY (patient_id) REFERENCES patients (patient_id)
        )
    ''')

    # Registros clínicos controlados
    cursor.executemany("INSERT INTO patients VALUES (?, ?, ?, ?, ?)", [
        (1, "John Doe", 62, "M", "2026-05-24"),
        (2, "Jane Roe", 45, "F", "2026-05-23"),
        (3, "Robert Smith", 67, "M", "2026-05-22")
    ])

    cursor.executemany("INSERT INTO medical_records VALUES (?, ?, ?, ?, ?, ?)", [
        (1, 1, "2026-05-24 10:00:00", "Loss of peripheral vision, tunnel vision effect.", "No pain, slow side vision decrease.", "Suspected Open-Angle Glaucoma"),
        (2, 2, "2026-05-23 14:30:00", "Frequent morning headaches and dizziness.", "High blood pressure spikes at home.", "Suspected Essential Hypertension")
    ])

    cursor.executemany("INSERT INTO exams VALUES (?, ?, ?, ?, ?, ?)", [
        (1, 1, "Tonometry (Intraocular Pressure Measurement)", "COMPLETED", "Elevated pressure: 24 mmHg", "2026-05-24"),
        (2, 1, "Visual Field Test (Perimetry)", "PENDING", None, "2026-05-24"),
        (3, 2, "Ambulatory Blood Pressure Monitoring (ABPM)", "COMPLETED", "24h average reading: 152/96 mmHg", "2026-05-23")
    ])

    conn.commit()
    conn.close()

# Inicializa fisicamente o banco
init_db()

# Cria o objeto de utilitário do LangChain apontando para o arquivo criado
db = SQLDatabase.from_uri("sqlite:////content/data.db")

@tool
def fetch_patient_clinical_context(patient_id: int) -> str:
    """Queries the hospital SQLite database to retrieve the full medical records, chief complaints, symptoms, and recent exam status for a specific patient ID."""
    try:
        patient_info = db.run(f"SELECT * FROM patients WHERE patient_id = {patient_id}")
        records = db.run(f"SELECT chief_complaint, symptoms, diagnostic_hypothesis FROM medical_records WHERE patient_id = {patient_id}")
        exams_status = db.run(f"SELECT exam_name, status, result FROM exams WHERE patient_id = {patient_id}")

        return f"=== PATIENT IDENTIFICATION ===\n{patient_info}\n\n=== CLINICAL HISTORY ===\n{records}\n\n=== RECENT EXAMS ===\n{exams_status}"
    except Exception as e:
        return f"Error retrieving database records: {str(e)}"

Overwriting hospital_project/database/connection.py


## 🧠 Etapa 3: Carregamento Otimizado da LLM Fine-Tuned

Escrevemos o módulo responsável por instanciar a nossa inteligência artificial customizada.

Utilizamos o framework Unsloth para carregar os adaptadores LoRA ajustados no treinamento, aplicando amostragem estável (temperature=0.6) e uma janela de contexto confortável (max_new_tokens=600) para evitar truncamento de texto, integrando o modelo diretamente a uma pipeline compatível com o LangChain.

In [ ]:
%%writefile hospital_project/models/custom_llm.py
from unsloth import FastLanguageModel
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

def load_custom_langchain_llm(caminho_model_drive):
    max_seq_length = 2048
    dtype = None
    load_in_4bit = True

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = caminho_model_drive,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=600,
        do_sample=True,
        temperature=0.6,
        repetition_penalty=1.2,
        top_p=0.9,
        return_full_text=False
    )

    return HuggingFacePipeline(pipeline=pipe)

Overwriting hospital_project/models/custom_llm.py


## 📝 Etapa 4: Subsistema de Logging e Segurança (Audit Trail)

Implementamos um mecanismo de rastreamento estrito. Toda transação feita pelo assistente clínico dispara uma gravação append-only no arquivo audit_trail.log.

O arquivo armazena os carimbos de data/hora, as queries dos médicos, as informações brutas fornecidas pelo banco de dados e a respectiva orientação gerada pela LLM.

In [ ]:
%%writefile hospital_project/utils/logger.py
from datetime import datetime

def log_clinical_transaction(patient_id: int, input_question: str, db_context: str, llm_response: str):
    log_path = '/content/audit_trail.log'
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    with open(log_path, 'a', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write(f"{timestamp} | INFO | AUDIT TRANSACTION STARTED FOR PATIENT ID: {patient_id}\n")
        f.write(f"DB CONTEXT RETRIEVED:\n{db_context}\n\n")
        f.write(f"USER QUESTION: {input_question}\n")
        f.write(f"SYSTEM RESPONSE GENERATED:\n{llm_response.strip()}\n")
        f.write(f"{timestamp} | INFO | AUDIT TRANSACTION COMPLETED\n")
        f.write("="*80 + "\n\n")

Overwriting hospital_project/utils/logger.py


## 🔌 Etapa 5: Alinhamento de Dependências e Setup do Servidor

Garantimos a homogeneidade do ambiente instalando as versões homologadas e travadas dos frameworks de IA, evitando conflitos de bibliotecas abertas e assegurando a portabilidade do código fonte.

In [ ]:
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --no-deps xformers==0.0.35 peft accelerate bitsandbytes trl
!pip install --no-cache-dir "transformers<5.6.0" "datasets<4.4.0"
!pip install --no-cache-dir langchain langchain-community langchain-huggingface grandalf

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l5trnug1/unsloth_5cfa64412b5a42b5b69d609c4eeea630
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l5trnug1/unsloth_5cfa64412b5a42b5b69d609c4eeea630
  Resolved https://github.com/unslothai/unsloth.git to commit af6504f900fe611a056e66eec6ab74976eab7f34
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 🛞 Etapa 6: Vinculação de Componentes e Teste de Importação

Adicionamos o projeto modularizado ao Path de execução do Python e realizamos os imports cruzados das ferramentas de dados, logs e inteligência para validar o isolamento de escopo antes da construção do agente.

In [ ]:

import sys
import os
import warnings
import pandas as pd
from google.colab import drive
from langchain_core.prompts import PromptTemplate

warnings.filterwarnings("ignore", category=FutureWarning)
drive.mount('/content/drive')

sys.path.append('/content/hospital_project')

from database.connection import fetch_patient_clinical_context, db
from models.custom_llm import load_custom_langchain_llm
from utils.logger import log_clinical_transaction

print("Todos os módulos customizados (.py) foram importados e validados com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Todos os módulos customizados (.py) foram importados e validados com sucesso!


## 🏗️ Etapa 7: Orquestração Baseada em Grafos de Estado (LangGraph)

Desenvolvemos uma máquina de estados finitos inteligente e segura usando o LangGraph. Em vez de permitir uma execução livre da LLM, o fluxo é rigidamente controlado por nós e arestas condicionais que implementam o protocolo de governança corporativa do hospital.

### 🧭 Lógica de Transição do Agente:
prontuario_medico (Nó inicial): Consulta a base de dados SQL.

* **router_pending_exams** (Roteador Condicional): Intercepta o prontuário. Se encontrar a string "PENDING", desvia o fluxo para proteção.

* **exames_pendentes** (Nó de Alerta): Injeta um aviso administrativo crítico forçando a equipe médica a priorizar exames antes de condutas definitivas.

* **build_prompt**: Concatena os contextos de forma limpa, aplicando os guardrails contra prescrições médicas diretas.

* **execute_llm**: Submete o input estruturado e seguro à nossa LLM refinada.

In [ ]:
!pip install grandalf

In [ ]:
# ==========================================
# 1) DEFINIÇÃO DO ESTADO COMPARTILHADO
# ==========================================
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

class ClinicalState(TypedDict, total=False):
    patient_id: int
    medical_question: str
    clinical_context: str
    has_pending_exams: bool
    alert_flag: str
    full_prompt: str
    final_response: str

# ==========================================
# 2) DEFINIÇÃO DOS NÓS (NODES)
# ==========================================

def node_fetch_database(state: ClinicalState) -> ClinicalState:
    """Busca o histórico e o contexto clínico do paciente no banco SQLite."""
    print(f"[NÓ] Buscando dados para o Paciente ID: {state['patient_id']}...")
    context = fetch_patient_clinical_context.invoke({"patient_id": state["patient_id"]})

    # Verifica de forma segura se existe a palavra 'PENDING' na string de exames retornada
    has_pending = "PENDING" in context

    return {
        "clinical_context": context,
        "has_pending_exams": has_pending,
        "alert_flag": "" # Inicializa vazio
    }

def node_generate_alert(state: ClinicalState) -> ClinicalState:
    """Nó ativado apenas se houver pendências. Injeta um aviso de governança."""
    print("[NÓ ALERT] Alerta crítico ativado: Paciente possui exames pendentes!")
    alert_msg = (
        "\nCRITICAL PROTOCOL ALERT: This patient has PENDING exams in their record. "
        "Prioritize validating the missing diagnostic results before finalizing clinical behavior."
    )
    return {"alert_flag": alert_msg}

def node_build_prompt(state: ClinicalState) -> ClinicalState:
    print("[NÓ] Construindo prompt final para o modelo...")

    rag_template = """You are an advanced medical virtual assistant. Your role is to analyze the provided clinical records and suggest the next investigative procedures for the medical staff.

CLINICAL ANALYSIS MANDATE:
1. Thoroughly review the patient metrics (such as the specific mmHg and visual complaints) and provide a comprehensive analysis of what this clinical state implies under standard medical protocols.
2. Address the medical team in the third person (e.g., 'The patient exhibits...'). Never address the patient directly.

SAFETY & GUARDRAILS:
- NEVER write a medical prescription, and NEVER mention specific drug dosages or treatment milligram hours to maintain proper compliance. Focus exclusively on diagnostic orientation and procedure recommendations.
{alert_zone}

### Patient Clinical Context:
{clinical_context}

### Staff Question:
{medical_question}

### Clinical Guidance Response:"""

    alert_content = state.get("alert_flag", "")

    # Se houver exames pendentes, o alerta entra aqui de forma limpa e clara
    if alert_content:
        alert_zone_msg = f"\n⚠️ ADMINISTRATIVE ALERT:\n{alert_content}\nInstruct the staff to prioritize scheduling or collecting these pending evaluations before any surgical or definitive therapeutic conduct.\n"
    else:
        alert_zone_msg = ""

    built_prompt = rag_template.format(
        alert_zone=alert_zone_msg,
        clinical_context=state["clinical_context"],
        medical_question=state["medical_question"]
    ).strip()

    return {"full_prompt": built_prompt}

def node_execute_llm(state: ClinicalState) -> ClinicalState:
    """Invoca o modelo customizado carregado via LangChain."""
    print("[NÓ LLM] Processando resposta via Llama-3 Fine-Tuned...")
    response = llm.invoke(state["full_prompt"])
    return {"final_response": response}

# ==========================================
# 3) FUNÇÃO ROTEADORA (CONDITIONAL EDGE)
# ==========================================

def router_pending_exams(state: ClinicalState) -> str:
    """Avalia o estado atual e decide se desvia para o nó de alerta ou vai direto."""
    if state.get("has_pending_exams", False):
        return "trigger_alert"
    return "go_to_prompt"

# ==========================================
# 4) CONSTRUÇÃO DO GRAFO (LANGGRAPH)
# ==========================================

# Inicializa o grafo com o nosso esquema de estado
workflow = StateGraph(ClinicalState)

# Adiciona todos os nós ao fluxo
workflow.add_node("prontuario_medico", node_fetch_database)
workflow.add_node("exames_pendentes", node_generate_alert)
workflow.add_node("build_prompt", node_build_prompt)
workflow.add_node("execute_llm", node_execute_llm)

# Define a porta de entrada padrão
workflow.set_entry_point("prontuario_medico")

# Adiciona as condicionais partindo da consulta do banco de dados
workflow.add_conditional_edges(
    "prontuario_medico",
    router_pending_exams,
    {
        "trigger_alert": "exames_pendentes",
        "go_to_prompt": "build_prompt"
    }
)

# Define as conexões diretas (arestas normais)
workflow.add_edge("exames_pendentes", "build_prompt") # Após o alerta, segue para o prompt
workflow.add_edge("build_prompt", "execute_llm")      # Do prompt, vai para a IA
workflow.add_edge("execute_llm", END)                 # Fim do processo

# Compila o grafo
app_clinical = workflow.compile()

# Exibe o mapa mental do fluxo gerado em formato ASCII para validação
print("\n=== ARQUITETURA DO GRAFO ===")
print(app_clinical.get_graph().draw_ascii())

# ==========================================
# 5) EXECUÇÃO ORQUESTRADA & AUDITORIA
# ==========================================

# Carrega o modelo de IA do Drive antes de rodar (garanta que a variável caminho_salvamento está definida)
caminho_salvamento = "/content/drive/MyDrive/Colab Notebooks/tech_challenge_fase3/models/lora_model"
llm = load_custom_langchain_llm(caminho_salvamento)

# Input de teste
initial_state = {
    "patient_id": 1,
    "medical_question": (
        "Based on the patient's records, analyze the clinical status regarding the suspected condition. "
        "Detail what this disease implies, what the current metrics indicate, and how the pending exams "
        "and scientific protocols help confirm the diagnosis."
    )
}

# Executa o Grafo de Estados
print("\nIniciando execução do Grafo...")
final_state_output = app_clinical.invoke(initial_state)

print("\n=== FINAL CLEAN RESPONSE FROM GRAFO ===")
print(final_state_output["final_response"].strip())
print("=" * 60)

# Grava na auditoria puxando os dados de dentro do estado final do grafo
log_clinical_transaction(
    patient_id=final_state_output["patient_id"],
    input_question=final_state_output["medical_question"],
    db_context=final_state_output["clinical_context"],
    llm_response=final_state_output["final_response"]
)
print("\n[SUCESSO] Transação de auditoria gravada em 'audit_trail.log'.")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



=== ARQUITETURA DO GRAFO ===
                  +-----------+          
                  | __start__ |          
                  +-----------+          
                        *                
                        *                
                        *                
              +-------------------+      
              | prontuario_medico |      
              +-------------------+      
                ...            ...       
              ..                  ..     
            ..                      ..   
+------------------+                  .. 
| exames_pendentes |                ..   
+------------------+              ..     
                ***            ...       
                   **        ..          
                     **    ..            
                +--------------+         
                | build_prompt |         
                +--------------+         
                        *                
                        *                
    

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



Iniciando execução do Grafo...
[NÓ] Buscando dados para o Paciente ID: 1...
[NÓ ALERT] Alerta crítico ativado: Paciente possui exames pendentes!
[NÓ] Construindo prompt final para o modelo...
[NÓ LLM] Processando resposta via Llama-3 Fine-Tuned...


Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=600) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== FINAL CLEAN RESPONSE FROM GRAFO ===
Suspected Open-Angle Glaucoma - Analysis:
=== OVERVIEW OF SUSPECTED CONDITION ===
Open-angle glaucoma occurs when fluid builds up too quickly inside your eye. It can damage the optic nerve at the back of your eyeball that carries signals from your eyes to your brain. The buildup happens because of changes within the drainage system of your eye called the trabecular meshwork. Normally, tiny blood vessels bring oxygen and nutrients to cells throughout your body including those found deep inside your eye. When you have open angle glaucoma, there may be more blood than usual flowing through these veins which causes them to bulge outwards like balloons filled with water instead of air. These enlarged capillaries block off normal flow so less liquid leaves your eyes via small openings between each cell layer making it harder for fluids inside your head space where they belong; eventually leading towards blindness if left untreated long enough without 

## 🪵 Etapa 8: Validação da Trilha de Auditoria (Compliance)

Para garantir que o sistema está registrando as interações de forma fidedigna e transparente, realizamos a leitura física do arquivo de logs gerado pela nossa última transação, provando a conformidade regulatória do assistente clínico.

In [ ]:
import os

print("=" * 60)
print("            PROOF OF AUDIT LOG REGISTRATION            ")
print("=" * 60)

if os.path.exists('/content/audit_trail.log'):
    with open('/content/audit_trail.log', 'r') as audit_file:
        log_content = audit_file.read()

        # Exibe as últimas linhas ou o bloco inteiro gerado
        print(log_content)
else:
    print("Error: 'audit_trail.log' not found in /content directory.")
print("=" * 60)

            PROOF OF AUDIT LOG REGISTRATION            
2026-05-25 14:00:22 | INFO | AUDIT TRANSACTION STARTED FOR PATIENT ID: 1
DB CONTEXT RETRIEVED:
=== PATIENT IDENTIFICATION ===
[(1, 'John Doe', 62, 'M', '2026-05-24')]

=== CLINICAL HISTORY ===
[('Loss of peripheral vision, tunnel vision effect.', 'No pain, slow side vision decrease.', 'Suspected Open-Angle Glaucoma')]

=== RECENT EXAMS ===
[('Tonometry (Intraocular Pressure Measurement)', 'COMPLETED', 'Elevated pressure: 24 mmHg'), ('Visual Field Test (Perimetry)', 'PENDING', None)]

USER QUESTION: Based on the patient's records, analyze the clinical status regarding the suspected condition. Detail what this disease implies, what the current metrics indicate, and how the pending exams and scientific protocols help confirm the diagnosis.
SYSTEM RESPONSE GENERATED:

2026-05-25 14:00:22 | INFO | AUDIT TRANSACTION COMPLETED

2026-05-25 14:27:47 | INFO | AUDIT TRANSACTION STARTED FOR PATIENT ID: 1
DB CONTEXT RETRIEVED:
=== PATIENT IDEN